In [2]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")
import fitz  # PyMuPDF
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from urllib.request import urlopen, Request

import warnings
warnings.filterwarnings('ignore')

In [3]:

files = {
    "jpmorgan_annual_report_2023.pdf": "https://www.jpmorganchase.com/content/dam/jpmc/jpmorgan-chase-and-co/investor-relations/documents/annualreport-2023.pdf",
    "fed_financial_stability_2024.pdf": "https://www.federalreserve.gov/publications/files/financial-stability-report-20241122.pdf",
    "basel_iii_framework.pdf": "https://www.bis.org/publ/bcbs189.pdf",
    "imf_macroeconomic_costs_conflict.pdf": "https://www.imf.org/-/media/files/publications/wp/2020/english/wpiea2020110-print-pdf.pdf",
    "imf_geopolitical_risks_2025.pdf": "https://www.imf.org/-/media/files/publications/gfsr/2025/april/english/ch2.pdf",
}

os.makedirs("../rag-docs", exist_ok=True)

for filename, url in files.items():
    filepath = os.path.join("../rag-docs", filename)
    
    if os.path.exists(filepath):
        print(f"Already exists: {filename}")
        continue
    
    print(f"Downloading {filename}...")
    try:
        req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(req, timeout=30) as response, open(filepath, "wb") as out_file:
            out_file.write(response.read())
        print(f"Saved: {filename}")
    except Exception as e:
        print(f"Failed: {filename} — {e}")

print("All done!")

Saved: jpmorgan_annual_report_2023.pdf
Saved: fed_financial_stability_2024.pdf
Saved: basel_iii_framework.pdf
Failed: imf_macroeconomic_costs_conflict.pdf — HTTP Error 403: Forbidden
Failed: imf_geopolitical_risks_2025.pdf — HTTP Error 403: Forbidden
All done!


the 2 forbidden docs were downloaded manually 

In [4]:
import os
files = os.listdir("../rag-docs")
print(f"Total files: {len(files)}")
for f in files:
    print(f)

Total files: 5
imf_macroeconomic_costs_conflict.pdf
fed_financial_stability_2024.pdf
basel_iii_framework.pdf
imf_geopolitical_risks_2025.pdf
jpmorgan_annual_report_2023.pdf


In [5]:
def load_docs(folder_path):
    documents = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            text = ""

            doc = fitz.open(file_path)
            for page in doc:
                text += page.get_text()
            doc.close()

            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "topic": filename.replace(".pdf", ""),
                        "source": file_path
                    }
                )
            )

    return documents


In [6]:
docs_before_split = load_docs("../rag-docs")
print(f"Loaded {len(docs_before_split)} documents.")

Loaded 5 documents.


In [17]:
# print(docs_before_split[0]) since the content is too long, i commenred it for better readability

In [8]:
len(docs_before_split[0].page_content)

71027

In [10]:
# The documents are too long, so we need to split them into smaller chunks.

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

doc_after_split = text_splitter.split_documents(docs_before_split)


In [11]:
len(doc_after_split[0].page_content)

924

In [13]:
# initializing the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {"device": "cpu"},
    encode_kwargs = {"normalize_embeddings": True}
)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10106.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# creating the vector database
vector_db = FAISS.from_documents(doc_after_split, embeddings)

In [15]:
# saving the vector database
vector_db.save_local("../database/faiss_index")
print("Vector database saved successfully.")

Vector database saved successfully.


In [36]:
# Load the saved index
vectorstore = FAISS.load_local("../database/faiss_index", embeddings, allow_dangerous_deserialization=True)

# Create retriever — fetch top 4 most relevant chunks
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})


In [37]:
import sys
sys.path.append("..")  # go up one level to find config.py
from config import llm

In [38]:
from langchain_core.prompts import ChatPromptTemplate

# Define the prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", """You are a senior financial analyst assistant. Answer questions using ONLY the provided context from financial documents.

        Guidelines:
        - Use ALL provided context to answer, even if the answer requires combining information from multiple documents
        - Only synthesize across documents when there is EXPLICIT evidence in BOTH sources — never infer relationships that are not directly stated
        - If the answer requires inference rather than direct quotes, prefix with "Based on available context, it appears that..."
        - If the answer truly cannot be found or inferred from the context, say 'I could not find this in the provided documents.'
        - Always cite which document your answer comes from using the document name in brackets
        - Include specific numbers, percentages, and figures when available
        - Never make up financial data or statistics
        - If combining information from multiple documents, clearly state which fact came from which document
        - Keep answers professional and concise

        Context: {context}"""),
                ("human", "{question}")
    ]
)

In [39]:
# Helper function to format retrieved documents for the prompt
def format_docs(docs):
    return "\n\n".join([f"[{doc.metadata.get('topic', 'unknown')}]\n{doc.page_content}" for doc in docs])



In [40]:
# Build the RAG chain
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [41]:
queries = [
    "What risks does JPMorgan highlight in their 2023 report?",
    "What are the main vulnerabilities in the US financial system?",
    "What is the countercyclical capital buffer?",
    "What is the minimum capital requirement under Basel III?",
    "What is Common Equity Tier 1 capital?",
    "What is the long term economic impact of civil war?",
    "What are the capital requirements banks must maintain during financial stress?",
    "How does the Fed's financial stability framework relate to Basel III requirements?"
]



for query in queries:
    print(f"\nQuestion: {query}")
    print("-" * 50)
    response = chain.invoke(query)
    print(f"Answer: {response}")
    print("=" * 50)


Question: What risks does JPMorgan highlight in their 2023 report?
--------------------------------------------------
Answer: According to the [jpmorgan_annual_report_2023], JPMorgan Chase highlights the following risks in their 2023 report:

1. Local, regional, and global business, economic, and calamities, including health emergencies, the spread of infectious diseases, epidemics or pandemics, an outbreak or escalation of hostilities or other geopolitical instabilities, the effects of climate change or extraordinary events beyond the Firm’s control [jpmorgan_annual_report_2023].
2. Ability of the Firm to maintain the security of its financial, accounting, technology, data processing, and other operational systems and facilities [jpmorgan_annual_report_2023].
3. Ability of the Firm to withstand disruptions that may be caused by any failure of its operational systems or those of third parties [jpmorgan_annual_report_2023].
4. Ability of the Firm to effectively defend itself against cy